In [56]:
# import packages
import pandas as pd
import numpy as np
import json, requests, os, dotenv
import geopy, folium, matplotlib

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.cluster import KMeans
from sklearn import linear_model

In [3]:
original_rating = pd.read_csv('location_rating.csv')
original_rating.head()

,location,rating
0,"Baltimore, Maryland, the United States",6
1,"Beijing, China",3
2,"Boston, Massachusetts, the United States",5
3,"Cambridge, the United Kingdom",8
4,"Chengdu, China",5


In [4]:
geolocator = geopy.geocoders.Nominatim(user_agent="foursquare_agent")

latitude_list = []
longitude_list = []

for address in original_rating['location']:
    try:
        location = geolocator.geocode(address, timeout = 10)
        if location:
            latitude_list.append(location.latitude)
            longitude_list.append(location.longitude)
        else:
            latitude_list.append(None)
            longitude_list.append(None)
    except geopy.exc.GeocoderTimedOut:
        latitude_list.append(None)
        longitude_list.append(None)
        print(f"Timeout: {address}")

original_rating['latitude'] = latitude_list
original_rating['longitude'] = longitude_list

In [50]:
# create map of Toronto using latitude and longitude values
map_original = folium.Map(location=[original_rating['latitude'].mean(), original_rating['longitude'].mean()], zoom_start=4)

# add markers to map
for lat, lng, label, rating in zip(original_rating['latitude'], original_rating['longitude'], original_rating['location'], original_rating['rating']):
    folium.CircleMarker([lat, lng], 
                        radius = 5,
                        tooltip =  f"{label}: {rating}",
                        color = 'blue', fill = True, fill_color = '#3186cc', fill_opacity = 0.7).add_to(map_original)  
map_original

In [15]:
# import the api_key for Foursquare
dotenv.load_dotenv("../personal_envs/neighborhood-preference-prediction.env", override=True)
api_key = os.getenv("toronto_venue_clustering")

In [16]:
# write the function of getting venues from Foursquare
def get_foursquare_raw_data(api_key, lat, long, radius, limit):
    url = "https://api.foursquare.com/v3/places/search"
    headers = {"Accept": "application/json", "Authorization": f"Bearer {api_key}"}
    params = {"ll": f"{lat},{long}", "radius": radius, "limit": limit, "sort": "DISTANCE"}
    response = requests.get(url, headers = headers, params = params)
    response.raise_for_status()
    return response.json()['results']

In [18]:
# get the venue list
venues_list = []

for location, lat, lng in zip(original_rating['location'], original_rating['latitude'], original_rating['longitude']):
    
    raw_data = get_foursquare_raw_data(api_key, lat, lng, 1000, 50)
    for v in raw_data:
        venues_list.append({
            "location": location,
            "fsq_id": v.get('fsq_id', None),
            "name": v.get('name', None),
            "lat": v['geocodes']['main']['latitude'] if 'geocodes' in v else None,
            "lon": v['geocodes']['main']['longitude'] if 'geocodes' in v else None,
            "distance": v.get('distance', None),
            "category": [c['name'] for c in v.get('categories', [])] if v.get('categories') else [],
            "location_address": v['location'].get('address', None),
            "location_locality": v['location'].get('locality', None),
            "location_region": v['location'].get('region', None),
            "location_postcode": v['location'].get('postcode', None),
            "location_country": v['location'].get('country', None),
            "location_formatted_address": v['location'].get('formatted_address', None),
            "timezone": v.get('timezone', None),
            "likely_open": v.get('closed_bucket', None)
        })

df = pd.DataFrame(venues_list)

In [21]:
# get the dummy data of category
mlb = MultiLabelBinarizer()
category_dummies = pd.DataFrame(mlb.fit_transform(df['category']), columns=mlb.classes_, index=df.index)
print(f'There are {len(category_dummies.columns)} uniques categories.')

rating_venues_dummy = pd.concat([df[['location']], category_dummies], axis=1)

There are 287 uniques categories.


In [68]:
# get the dummy data grouped
rating_venues_dummy_group = rating_venues_dummy.groupby('location').mean().reset_index()
rating_venues_dummy_group = rating_venues_dummy_group.merge(original_rating[['location', 'rating']], on='location', how='inner')

rating_venues_dummy_group.head(3)

,location,Acupuncture Clinic,Advertising Agency,American Restaurant,Amphitheater,Amusement Park,Arcade,Architecture Firm,Art Gallery,Art Museum,...,Wine Bar,Wings Joint,Women's Store,Yakitori Restaurant,Yoshoku Restaurant,Youth Organization,Zhejiang Restaurant,Zoo,Zoo Exhibit,rating
0,"Baltimore, Maryland, the United States",0.0,0.041667,0.00,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,6
1,"Beijing, China",0.0,0.000000,0.00,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,3
2,"Boston, Massachusetts, the United States",0.0,0.000000,0.02,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.02,0.0,0.0,0.0,5


In [38]:
# get the place for prediction
place_test = 'Athens, Greece'
test_data = pd.DataFrame([[place_test]] , columns = ['location']) 

geolocator = geopy.geocoders.Nominatim(user_agent="foursquare_agent")

latitude_list = []
longitude_list = []

for address in test_data['location']:
    try:
        location = geolocator.geocode(address, timeout = 10)
        if location:
            latitude_list.append(location.latitude)
            longitude_list.append(location.longitude)
        else:
            latitude_list.append(None)
            longitude_list.append(None)
    except geopy.exc.GeocoderTimedOut:
        latitude_list.append(None)
        longitude_list.append(None)
        print(f"Timeout: {address}")

test_data['latitude'] = latitude_list
test_data['longitude'] = longitude_list

In [41]:
# get the venue list
venues_list = []

for location, lat, lng in zip(test_data['location'], test_data['latitude'], test_data['longitude']):
    
    raw_data = get_foursquare_raw_data(api_key, lat, lng, 1000, 50)
    for v in raw_data:
        venues_list.append({
            "location": location,
            "fsq_id": v.get('fsq_id', None),
            "name": v.get('name', None),
            "lat": v['geocodes']['main']['latitude'] if 'geocodes' in v else None,
            "lon": v['geocodes']['main']['longitude'] if 'geocodes' in v else None,
            "distance": v.get('distance', None),
            "category": [c['name'] for c in v.get('categories', [])] if v.get('categories') else [],
            "location_address": v['location'].get('address', None),
            "location_locality": v['location'].get('locality', None),
            "location_region": v['location'].get('region', None),
            "location_postcode": v['location'].get('postcode', None),
            "location_country": v['location'].get('country', None),
            "location_formatted_address": v['location'].get('formatted_address', None),
            "timezone": v.get('timezone', None),
            "likely_open": v.get('closed_bucket', None)
        })

test_df = pd.DataFrame(venues_list)

In [45]:
# get the dummy data of category
mlb = MultiLabelBinarizer()
test_category_dummies = pd.DataFrame(mlb.fit_transform(test_df['category']), columns=mlb.classes_, index = test_df.index)
print(f'There are {len(test_category_dummies.columns)} uniques categories.')

test_rating_venues_dummy = pd.concat([test_df[['location']], category_dummies], axis=1)

There are 44 uniques categories.


In [46]:
# get the dummy data grouped
test_rating_venues_dummy_group = test_rating_venues_dummy.groupby('location').mean().reset_index()
test_rating_venues_dummy_group.head(3)

,location,Acupuncture Clinic,Advertising Agency,American Restaurant,Amphitheater,Amusement Park,Arcade,Architecture Firm,Art Gallery,Art Museum,...,Whisky Bar,Wine Bar,Wings Joint,Women's Store,Yakitori Restaurant,Yoshoku Restaurant,Youth Organization,Zhejiang Restaurant,Zoo,Zoo Exhibit
0,"Athens, Greece",0.0,0.02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [54]:
# match the test columns to original
test_rating_venues_dummy_group = test_rating_venues_dummy_group.reindex(
    columns=rating_venues_dummy_group.columns,
    fill_value=0
)

In [55]:
test_rating_venues_dummy_group

,location,Acupuncture Clinic,Advertising Agency,American Restaurant,Amphitheater,Amusement Park,Arcade,Architecture Firm,Art Gallery,Art Museum,...,Whisky Bar,Wine Bar,Wings Joint,Women's Store,Yakitori Restaurant,Yoshoku Restaurant,Youth Organization,Zhejiang Restaurant,Zoo,Zoo Exhibit
0,"Athens, Greece",0.0,0.02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [81]:
lr = linear_model.LinearRegression()
x = np.asanyarray(rating_venues_dummy_group[rating_venues_dummy_group.columns[1:-1]])
y = np.asanyarray(rating_venues_dummy_group[rating_venues_dummy_group.columns[-1]])
lr.fit (x, y)

# The prediction of the test place
test_prediction= lr.predict(np.asanyarray(test_rating_venues_dummy_group[rating_venues_dummy_group.columns[1:-1]]))
print(f'The predicted rate of {place_test} will be {round(test_prediction[0], 2)}.')

The predicted rate of Athens, Greece will be 4.55.
